<a href="https://colab.research.google.com/github/jdasam/aat3020/blob/2026/NLP_Assignment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 1

In this assignment, you will explore word vectors — what they encode, how they can be manipulated, and what they reveal about language and culture.

**Problems overview:**
1. Find Most Similar Words
2. Word Analogy
3. Semantic Axis — project words onto a meaningful direction in vector space
4. Word Walk — travel step by step from one concept to another
5. Bias Probe — uncover what associations are encoded in the model
6. Visualize Word Vectors
7. Train Your Own Word2Vec

**Submission:** A report in `pdf`, your completed notebook file in `ipynb`, and training data in `txt` (for Problem 7)
- The assignment will be evaluated mainly on the report. Include every result you want to present, including figures.
- Report: Free format
  - If you are native Korean speaker, you HAVE to write your report in Korean.
    한국어가 모국어인 경우, 레포트는 꼭 한국어로 작성하여야 합니다. 영어로 작성하고 싶은 경우, 작문에 있어서 LLM의 교정을 받지 않아야만 합니다.
  - If you are not native Korean speaker, you can write your report in English.
- Save your notebook with cell outputs and submit as `ipynb`

**Evaluation criteria:**
- How interesting and original are your examples and observations
- How well you explain *why* results turn out the way they do, in light of how Word2Vec is trained
- Any description suspicious of uncomprehended LLM use may be penalized.

## 0. Setup
- Check ``gensim`` library is installed
  - if not, you can install using ``!pip install gensim``
- List the downloadable vectors from ``gensim``


In [ ]:
!pip install gensim
import gensim
import numpy as np
import pprint as pp

In [ ]:
import gensim.downloader
list(gensim.downloader.info()['models'].keys())

- Among the Word2Vec model codes above, select one model of your choice among ``glove-wiki-gigaword`` or ``glove-twitter``
    - numbers at the last represents the number of dimension of each Word2Vec Model
        - e.g. ``glove-twitter-200`` was trained on twitter dataset while embedding each word into 200-dim vector
        - e.g. ``glove-wiki-gigaword-300`` was trained on wikipedia dataset while embedding each word into 300-dim vector
    - Model with higher dimension will show more accurate result in the following problems.
- Download the selected model and load it as a ``model``

In [ ]:
your_model_code = 'glove-wiki-gigaword-300' # select among the model code aboves
model = gensim.downloader.load(your_model_code) # download and load the model. It can take some time

In [ ]:
# test the model output
model['cat']

## Warm-up: Getting Familiar with the Model

Before diving into the problems, let's explore what the model can do.

- `model[word]` — returns the word's vector (a numpy array with hundreds of floats)
- `model.has_index_for(word)` — checks if a word is in the vocabulary
- `model.most_similar(word)` — returns the 10 most similar words
- `model.similarity(word_a, word_b)` — returns cosine similarity between two words (between −1 and 1)
- `model.most_similar(positive=[word_a, word_b], negative=[word_c])` — does vector arithmetic

Run the cells below and try changing the words to something you're curious about.


In [ ]:
# Try changing this word to something you're curious about
word = 'music'
assert model.has_index_for(word), f"'{word}' is not in the vocabulary — try a different word"

print(f"Vector for '{word}': {len(model[word])} dimensions\n")
print(f"10 most similar words to '{word}':")
for w, score in model.most_similar(word):
    print(f"  {w:20s}  similarity: {score:.4f}")

In [ ]:
# Compare cosine similarity between word pairs — closer to 1 means more similar
pairs = [
    ('cat',  'dog'),
    ('cat',  'car'),
    ('good', 'bad'),
    ('king', 'queen'),
]
print("Cosine similarity between word pairs:\n")
for a, b in pairs:
    print(f"  '{a}' ↔ '{b}': {model.similarity(a, b):.4f}")

## Problem 1. Find Most Similar Words

### Part A. Most Similar
- One of the most simple and typical use case of Word2Vec is finding a word based on similarity.
- You can list the most similar words for a given query word by using ``model.most_similar(your_word)``
    - Usually, every word in Word2Vec model is in lowercase
- **In your report**, present **3** interesting examples and explain **why it was interesting for you**
    - Try to explain why those words are regarded similar in Word2Vec, considering how it was trained
- Caution: The model was trained with multilingual dataset. This means the meaning of the word can follow non-English words.
    - e.g. "die", "war" are more frequently used in German than English.

### Part B. Doesn't Match
- ``model.doesnt_match(word_list)`` takes a list of words and returns the word that is least similar to the others.
    - e.g. ``model.doesnt_match(['breakfast', 'lunch', 'dinner', 'cereal'])`` → ``'cereal'``
- **Design 3 word lists of your own** where you predict which word should be the odd one out.
    - Include both cases where the model **succeeds** and where it **fails** if possible.
- **In your report**, for each example:
    - State which word you expected to be the odd one out and why
    - Explain whether the model was correct, and why you think it succeeded or failed

In [ ]:
### Part A: Most Similar (present 3 interesting examples)

target_word = 'sogang' # Enter your word string here
assert model.has_index_for(target_word), f"The selected word, {target_word}, is not included in the model's vocabulary"

results = model.most_similar(target_word)
header = f" Top 10 words most similar to '{target_word}' "
col_w = max(len(w) for w, _ in results) + 2

print(f"\n  ╔{'═' * (col_w + 2)}╦{'═' * 18}╗")
print(f"  ║{header:^{col_w + 2}}║{'  similarity  ':^18}║")
print(f"  ╠{'═' * (col_w + 2)}╬{'═' * 18}╣")
for i, (word, score) in enumerate(results, 1):
    bar = '█' * round(score * 12) + '░' * (12 - round(score * 12))
    print(f"  ║ {i:>2}. {word:<{col_w - 4}}  ║  {bar}  {score:.4f} ║")
print(f"  ╚{'═' * (col_w + 2)}╩{'═' * 18}╝\n")

In [ ]:
### Part B: Doesn't Match (design 3 word lists of your own)

# Example — replace with your own word lists
word_list_1 = ['breakfast', 'lunch', 'dinner', 'cereal']
word_list_2 = ['cat', 'dog', 'fish', 'piano']
word_list_3 = ['paris', 'berlin', 'london', 'germany']

for i, wl in enumerate([word_list_1, word_list_2, word_list_3], 1):
    for word in wl:
        assert model.has_index_for(word), f"'{word}' is not in the model's vocabulary"
    result = model.doesnt_match(wl)
    print(f"List {i}: {wl}")
    print(f"  → Odd one out: '{result}'\n")

## Problem 2. Word Analogy
- Word analogy is one of the most famous demonstrations of word vectors.
- The idea: if you subtract one word's vector from another and add a third, you get something close to the "answer."
- `analogy(model, 'man', 'king', 'woman')` asks: **"man is to king as woman is to ____?"**
    - It computes: `normalize(king) − normalize(man) + normalize(woman)` and finds the nearest word.

**Tip — how to read the function signature** `analogy(model, x1, x2, y1)`:
- The relation goes **x1 → x2**, and you want the same relation applied to **y1 → ?**
- ✅ `analogy(model, 'man', 'king', 'woman')` → "man is to king as woman is to ___"
- ✅ `analogy(model, 'france', 'paris', 'japan')` → "France is to Paris as Japan is to _
- **In your report**, present at least **3** interesting examples of your choice
    - Include both successes and failures — failures are often just as revealing
    - Describe what you expected and why the result was interesting (or surprising)

In [ ]:
def analogy(model, x1, x2, y1):
    results = model.most_similar(positive=[x2, y1], negative=[x1])
    col_w = max(len(w) for w, _ in results) + 2

    question = f"  {x1}  :  {x2}   =   {y1}  :  ???"
    print(f"\n{question}")
    print(f"  {'─' * (len(question) - 2)}")
    print(f"  ╔{'═' * (col_w + 2)}╦{'═' * 18}╗")
    print(f"  ║{'  candidates ':^{col_w + 2}}║{'  similarity  ':^18}║")
    print(f"  ╠{'═' * (col_w + 2)}╬{'═' * 18}╣")
    for i, (word, score) in enumerate(results, 1):
        marker = ' ◀ best' if i == 1 else ''
        bar = '█' * round(score * 12) + '░' * (12 - round(score * 12))
        print(f"  ║ {i:>2}. {word:<{col_w - 4}}  ║  {bar}  {score:.4f} ║{marker}")
    print(f"  ╚{'═' * (col_w + 2)}╩{'═' * 18}╝\n")
    print(f"  ✦  Answer:  {x1} : {x2} = {y1} : '{results[0][0]}'\n")

# Try with your own word choice
analogy(model, 'man', 'king', 'woman')

## Problem 3. Semantic Axis (Word Compass)

So far, we've been looking for the single "nearest" word to a query. But word vectors live in a **continuous space** — we can do much more.

A **semantic axis** is a direction in vector space defined by two contrasting words. By projecting a set of words onto this axis, you can measure how strongly each word leans toward one pole.

**How it works:**

Given pole words A and B:
- Axis direction = `normalize(B) − normalize(A)`
- Score for word W = `dot(normalize(W), axis)`
- Positive score → leans toward B; Negative → leans toward A

### Your Task
- Use `plot_semantic_axis` to explore **at least 2 semantic axes of your own design**
- Pick word lists that reveal something interesting, surprising, or thought-provoking
- **In your report**: Did the results match your intuition? What surprised you?

**Suggestions:**
- Aesthetic: `"ugly" ↔ "beautiful"`, `"noisy" ↔ "silent"`, `"chaotic" ↔ "ordered"`
- Temporal: `"ancient" ↔ "modern"`, `"slow" ↔ "fast"`, `"past" ↔ "future"`
- Cultural: `"eastern" ↔ "western"`, `"formal" ↔ "casual"`, `"urban" ↔ "rural"`
- Design axes that feel personally meaningful to you

In [ ]:
def plot_semantic_axis(model, pole_a, pole_b, words):
    """
    Plots words on a spectrum from pole_a (left) to pole_b (right).
    Words with close x positions are staggered vertically to avoid label overlap.
    """
    missing = [w for w in words if not model.has_index_for(w)]
    if missing:
        print(f"Skipping words not in vocabulary: {missing}")
    words = [w for w in words if model.has_index_for(w)]

    vec_a = model[pole_a] / np.linalg.norm(model[pole_a])
    vec_b = model[pole_b] / np.linalg.norm(model[pole_b])
    axis = vec_b - vec_a

    scores = {w: float(np.dot(model[w] / np.linalg.norm(model[w]), axis)) for w in words}
    sorted_items = sorted(scores.items(), key=lambda x: x[1])

    # Assign staggered y levels to avoid label overlap
    x_vals = [s for _, s in sorted_items]
    score_range = (max(x_vals) - min(x_vals)) if len(x_vals) > 1 else 1.0
    min_gap = score_range * 0.08

    y_levels = [0] * len(sorted_items)
    for i in range(1, len(sorted_items)):
        if sorted_items[i][1] - sorted_items[i - 1][1] < min_gap:
            y_levels[i] = (y_levels[i - 1] % 3) + 1
        else:
            y_levels[i] = 0
    n_levels = max(y_levels) + 1

    import plotly.graph_objects as go
    colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A',
              '#19D3F3', '#FF6692', '#B6E880', '#FF97FF', '#FECB52']
    fig = go.Figure()
    for i, ((word, score), y_lvl) in enumerate(zip(sorted_items, y_levels)):
        fig.add_trace(go.Scatter(
            x=[score], y=[y_lvl * 0.3],
            mode='markers+text',
            text=[word], textposition='top center',
            marker=dict(size=10, color=colors[i % len(colors)]),
            showlegend=False
        ))
    fig.update_layout(
        title=f'Semantic Axis: "{pole_a}" ←——→ "{pole_b}"',
        xaxis_title=f'← {pole_a}   |   {pole_b} →',
        yaxis=dict(visible=False, range=[-0.3, n_levels * 0.3 + 0.5]),
        height=max(320, 240 + n_levels * 40),
    )
    fig.show()

    print(f"\nRanking  (← {pole_a}  to  {pole_b} →):")
    for word, score in sorted_items:
        print(f"  {score:+.3f}  {word}")

In [ ]:
# Example: where do art-related words fall between "abstract" and "realistic"?
art_words = ['impressionism', 'cubism', 'photography', 'sculpture', 'portrait',
             'sketch', 'animation', 'graffiti', 'calligraphy', 'collage', 'painting']

plot_semantic_axis(model, 'abstract', 'realistic', art_words)

In [ ]:
# Your turn — design your own semantic axis
pole_a = 'ancient'   # ← change me
pole_b = 'modern'    # ← change me

your_words = ['jazz', 'rock', 'classical', 'electronic', 'folk', 'opera', 'rap', 'blues', 'punk', 'ambient']

plot_semantic_axis(model, pole_a, pole_b, your_words)

## Problem 4. Word Walk (Traveling Through Vector Space)

Imagine taking a step-by-step journey from one word to another, through the space of all words. At each step, you find the closest word to your current position.

This "word walk" can reveal unexpected conceptual bridges between ideas — and sometimes produce poetic or surprising detours.

**How it works:**

We linearly interpolate between `start` and `end`:

`v(t) = (1 − t) × vector(start) + t × vector(end)`

At each step, we find the nearest word to `v(t)`.

### Your Task
- Explore **at least 2 word walks** of your own choosing
- Look for walks that produce interesting, surprising, or poetic intermediate steps
- **In your report**: Describe the journey. What did you discover along the way?

**Suggestions:**
- Emotions: `"love" → "hate"`, `"joy" → "grief"`, `"fear" → "courage"`
- Concepts: `"war" → "peace"`, `"nature" → "technology"`, `"silence" → "noise"`
- What two words would make the most interesting journey?

In [ ]:
from sklearn.decomposition import PCA

# In 300-D space most word vectors are nearly equidistant from each other
# (curse of dimensionality), so interpolating between two endpoints keeps
# pulling up morphological variants of the same word (loves, loving, hatred…).
# Projecting to a lower-dimensional PCA space compresses semantic structure
# and yields much more diverse, interesting intermediate words.
_VOCAB_SIZE   = 50_000   # use the top-N most frequent words
_N_COMPONENTS = 50       # PCA target dimension

print(f"Fitting PCA on top {_VOCAB_SIZE:,} words  ({model.vector_size}D → {_N_COMPONENTS}D) …")
_ww_vocab   = list(model.key_to_index.keys())[:_VOCAB_SIZE]
_ww_matrix  = np.array([model[w] for w in _ww_vocab], dtype=np.float32)
_ww_pca     = PCA(n_components=_N_COMPONENTS, random_state=0)
_ww_reduced = _ww_pca.fit_transform(_ww_matrix)   # (vocab_size, 50)
_ww_w2i     = {w: i for i, w in enumerate(_ww_vocab)}
print(f"Done.  Explained variance captured: {_ww_pca.explained_variance_ratio_.sum():.1%}\n")


def word_walk(model, start, end, steps=10):
    """
    Linearly interpolates between `start` and `end` in PCA-compressed space
    and shows the nearest (unseen) word at each step.
    """
    for w in (start, end):
        assert model.has_index_for(w), f"'{w}' not in model vocabulary"
        assert w in _ww_w2i,           f"'{w}' not in top-{_VOCAB_SIZE:,} PCA vocabulary"

    v1 = _ww_reduced[_ww_w2i[start]]
    v2 = _ww_reduced[_ww_w2i[end]]

    print(f"  '{start}'  ──────────────────────────────────────►  '{end}'")
    print(f"  ({model.vector_size}D → {_N_COMPONENTS}D PCA  ·  {steps} steps)\n")

    seen = {start, end}
    for t in np.linspace(0, 1, steps):
        v    = (1 - t) * v1 + t * v2
        dists = np.linalg.norm(_ww_reduced - v, axis=1)
        word = '—'
        for idx in np.argsort(dists):
            w = _ww_vocab[idx]
            if w not in seen:
                word = w
                seen.add(w)
                break
        bar = '█' * int(t * 36) + '░' * (36 - int(t * 36))
        print(f"  |{bar}|  {word}")
    print()

In [ ]:
# Try a word walk
word_walk(model, 'love', 'hate', steps=12)

In [ ]:
# Your turn — what journey do you want to take?
word_walk(model, 'nature', 'technology', steps=12)

## Problem 5. Bias Probe

Word vectors are trained on human-written text — which means they inherit the **biases, associations, and assumptions** of the people who wrote it. This includes gender roles, cultural stereotypes, and social hierarchies.

By projecting words onto a semantic axis like `"woman" ↔ "man"` or `"poor" ↔ "rich"`, we can literally measure what associations are encoded in the model.

This is not just a technical curiosity. It connects to fundamental questions about AI, representation, and power — central themes in art & technology.

### Your Task
- Use `plot_semantic_axis` to probe **at least 2 types of bias** of your own choice
- **In your report**: What biases did you find? Were they expected or surprising? What does this imply about AI systems trained on such data?

**Suggestions:**
- Gender and professions: project job titles onto a `"woman" ↔ "man"` axis
- Class: project social/cultural words onto `"poor" ↔ "rich"` or `"rural" ↔ "urban"`
- Technology: project concepts onto `"human" ↔ "machine"` or `"emotion" ↔ "logic"`
- Design a probe that you personally find meaningful or troubling

In [ ]:
# Example: Do job titles have a gender association in the model?
professions = ['nurse', 'engineer', 'teacher', 'ceo', 'artist', 'scientist',
               'doctor', 'secretary', 'programmer', 'designer', 'chef', 'pilot']

plot_semantic_axis(model, 'woman', 'man', professions)

In [ ]:
# Your turn — design your own bias probe
pole_a = 'poor'   # ← change me
pole_b = 'rich'   # ← change me

your_words = ['vacation', 'university', 'hospital', 'museum', 'subway',
              'restaurant', 'garden', 'library', 'mall', 'stadium', 'prison', 'yacht']

plot_semantic_axis(model, pole_a, pole_b, your_words)

## Problem 6. Visualize Word Vectors

Word vectors live in a high-dimensional space (100–300 dimensions), which is impossible to visualize directly. PCA compresses all those dimensions down to 2, preserving as much structure as possible — so words that were nearby in high-D space tend to stay nearby in 2D.

To make the map more readable and interesting, use **`display_pca_scatterplot_grouped`**: it takes a dictionary of themed word groups and color-codes each group in the scatter plot.

### Your Task
- Design **at least 3 themed word groups** with a total of **30+ words**
- Each group should have a clear theme (e.g., emotions, colors, music genres, art movements, countries, body parts, foods…)
- **In your report**, describe what you see:
    - How are the words clustered? Do words from the same group stay together?
    - Are there any words that ended up far from their group — or surprisingly close to another group?
    - What does the 2D layout reveal about the relationships between concepts?

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import plotly.express as px
import pandas as pd

def display_pca_scatterplot_grouped(model, word_groups):
    """
    Visualizes word vectors with PCA, color-coded by group.
    word_groups: dict of {group_name: [list of words]}
    """
    all_words, all_groups = [], []
    for group_name, words in word_groups.items():
        for w in words:
            if model.has_index_for(w):
                all_words.append(w)
                all_groups.append(group_name)
            else:
                print(f"  Warning: '{w}' not in vocabulary — skipping")

    if len(all_words) < 30:
        print(f"WARNING: For your report, use 30+ words total. Current count: {len(all_words)}")

    word_vectors = np.array([model[w] for w in all_words])
    twodim = PCA().fit_transform(word_vectors)[:, :2]

    df = pd.DataFrame({'x': twodim[:, 0], 'y': twodim[:, 1], 'word': all_words, 'group': all_groups})
    fig = px.scatter(df, x='x', y='y', text='word', color='group',
                     title='Word Vectors — PCA (colored by group)')
    fig.update_traces(textposition='top center')
    fig.show()



In [ ]:
# Organize words into themed groups — replace these with themes that interest YOU
word_groups = {
    'emotions':    ['joy', 'anger', 'fear', 'sadness', 'love', 'anxiety', 'peace', 'pride'],
    'colors':      ['red', 'blue', 'green', 'yellow', 'purple', 'orange', 'black', 'white'],
    'art_styles':  ['impressionism', 'cubism', 'baroque', 'surrealism', 'minimalism', 'abstract'],
    'music_genres':['jazz', 'classical', 'rock', 'electronic', 'folk', 'opera', 'blues', 'punk'],
}

display_pca_scatterplot_grouped(model, word_groups)

## Problem 7. Train New Word2Vec
- Word2Vec models can be trained on different corpus (text)
- Train your own model with your custom selection of text
- Use the functions from previous problems (`most_similar`, word analogy, `plot_semantic_axis`, etc.) freely to explore your trained model
- In your report, present at least **3** interesting examples that make different results by dataset selection
    - Feel free to reuse any function introduced in earlier problems — find whichever examples you think are most surprising or revealing
    - You don't have to repeat all the analysis again. Select some examples that you think are interesting
- Explain the difference of the result by dataset selection
- You can refer [Official Documentation](https://radimrehurek.com/gensim/models/word2vec.html#gensim.models.word2vec.Word2Vec) Word2Vec Model

In [2]:
# You don't have to change this cell
import string
from gensim.models import Word2Vec

def remove_punctuation(x):
  return x.translate(''.maketrans('', '', string.punctuation))
def make_tokenized_corpus(corpus):
  out= [ [y.lower() for y in remove_punctuation(sentence).split(' ') if y] for sentence in corpus]
  return [x for x in out if x!=[]]

In [ ]:
your_text_fn = '' # Enter your text file name here

with open(your_text_fn, 'r') as f:
  strings = f.readlines()

'''
This line is for the case when the text file is not properly formatted.
It was used to ignore linebreaks and join the sentences into one string, since the text example included linebreak following printed book lines.

strings = "".join(strings).replace('\n', ' ').replace('Mr.', 'mr').replace('Mrs.', 'mrs').split('. ')
'''
# The strings has to be a list of list of strings, where inner list is a sentence

print("Checking the first 5 sentences in the text file")
for i in range(5):
  print(f"Sentence {i+1}: {strings[i]}")
corpus = make_tokenized_corpus(strings)

- gensim Word2Vec arguments
  - ``sentences``: list of list of strings
  - ``vector_size``: dimension of word vector
  - ``epochs``: number of epoch to train the word2vec model
  - ``window``: maximum distance between the current and predicted word within a sentence
  - ``min_count``: ignore all words with total frequency lower than this
  - ``sg``: training algorithm: 1 for skip-gram; otherwise CBOW
  - ``negative``: if > 0, negative sampling will be used, the int for negative specifies how many "noise words" should be drown (usually between 5-20)

In [20]:
model = Word2Vec(sentences=corpus, vector_size=200, window=5, min_count=2, epochs=50, sg=1)
model = model.wv # To match with previous codes, we use wv (KeyedVector) of the Word2Vec class
# Try the function above with the newely trained model